In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
import ana_utils
import filter_utils

In [ ]:
from filter_utils import filter_params, plot_with_hist

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

## Data loading / setup

In [ ]:
obj = "yasone1"

In [ ]:
cat = ana_utils.read_catalogue(obj, filter_bad=False, catname="allcolours")

In [ ]:
cat_psf = ana_utils.read_catalogue(obj, filter_bad=False, catname="allcolours_psf")
cat_julen = ana_utils.read_catalogue(obj, filter_bad=False, catname="allcolours_julen")
cat_julen_stack = ana_utils.read_catalogue(obj, filter_bad=False, catname="allcolours_julen_stack")
cat_julen_sep= ana_utils.read_catalogue(obj, filter_bad=False, catname="julen")

In [ ]:
obs_props = ana_utils.get_obs_props(obj)

In [ ]:
filter_utils.calibrate_mag_col(cat_psf, "MAG_PSF")

In [ ]:
for filt in ["G", "R", "I"]:
    cat_psf.rename_column(f"{filt}_MAGERR_PSF", f"{filt}_MAG_PSF_ERR")



In [ ]:
coord0 = SkyCoord(obs_props["ra"], obs_props["dec"], unit=u.degree)

## Extra plot utils

In [ ]:
def plot_tangent(cat):
    plt.scatter(cat["xi"], cat["eta"], s=1, lw=0, color="k")
    plt.xlabel("xi / arcmin")
    plt.ylabel("eta / arcmin")
    plt.gca().set_aspect(1)
    plt.gca().invert_xaxis()


In [ ]:
def get_default_params(objname=obj, **kwargs):
    obs_props = ana_utils.get_obs_props(objname)

    default_kwargs = dict(dm = obs_props["dm"], 
        iso_fe_h = obs_props["iso_fe_h"],
        iso_log_age = np.log10(obs_props["iso_age"]),
        objname = objname, 
        mode="gri", 
        A_V = 0.5,
        r23_max_sigma=3,
                         )

    return filter_utils.filter_params(
        **(default_kwargs | kwargs)
    )
    

In [ ]:
def plot_tangent_filter_type(cat, params):
    fig, axs = plt.subplots(2, 2, figsize=(4, 4), sharex=True, sharey=True)

    plt.sca(axs[0][0])
    plot_tangent(cat)
    plt.title("default")

    plt.sca(axs[0][1])
    filt_flag = filter_utils.get_flags_filter(cat, params)
    plot_tangent(cat[~filt_flag])
    plt.title("flagged")

    plt.sca(axs[1][0])
    filt = filter_utils.get_star_galaxy_filter(cat, params)
    plot_tangent(cat[filt_flag & filt])
    plt.title("stars")

    plt.sca(axs[1][1])
    plot_tangent(cat[filt_flag & ~filt])
    plt.title("galaxies")


In [ ]:
params = get_default_params(obj)
filt = filter_utils.get_star_galaxy_filter(cat, params)

plt.scatter(cat["R_MAG"][filt], cat["MAG_23"][filt], s=1, label="star?", lw=0)
plt.scatter(cat["R_MAG"][~filt], cat["MAG_23"][~filt], s=1, label="galaxy?", lw=0)

arya.Legend(-1)
plt.axhline(filter_utils.derive_threshhold(cat["MAG_23"], cat), color="k")

plt.xlabel("R mag")
plt.ylabel("mag(3px) - mag(5px)")
plt.xlim(17, 27)
plt.ylim(-0.5, 1.5)

In [ ]:
params = get_default_params(obj)
params.r23_max_sigma = 3
plot_tangent_filter_type(cat, params)

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(5, 4), sharex=True, sharey=True)

plt.sca(axs[0][0])
plot_tangent(cat)
plt.title("default")

plt.sca(axs[0][1])
plot_tangent(cat_psf)
plt.title("psf")

plt.sca(axs[0][2])
plot_tangent(cat_julen)
plt.title("julen")


plt.sca(axs[1][2])
plot_tangent(cat_julen_stack)
plt.title("julen stacked")


plt.sca(axs[1][1])
plot_tangent(cat_julen_sep)
plt.title("julen sep")

axs[1][0].remove()

plt.tight_layout()


We want to sample random positions which we can place complete circles. 
The below code uses a polyhedral region around the target field and excludes regions nearby the centre (where we measure),
nearby bright stars (for Yasone1) and nearby the edges.

In [ ]:
params_ri = get_default_params(mode="ri")
params_ri_sgs = get_default_params(mode="ri", r23_max_sigma=None)
params_gr = get_default_params(mode="gr")
params_gri = get_default_params(mode="gri")
params_n = get_default_params(mode="n")

In [ ]:
1 / filter_utils.catalog_stats(cat, params_n).area_scale

In [ ]:
filter_utils.plot_2d_counts(cat, 20/60, thresh=36)

In [ ]:
plot_with_hist(cat, params_n)

The plot above is our detection diagnostic plot for an unfiltered sample. Each panel (in reading order) is
1. Top left: the stars excluded by the present selection criteria (black) on the tangent plane. The blue and orange circles represent the example central and background regions.
2. Top middle left: The g-r CMD for stars within the central region. Blue pointss are selected by the filter criterion and black are not.
3. Top middle right: Same as 2 but for r-i
4. Top right: The probability distribution of the counts of selected stars within background regions. The blue histogram is from randomly selected regions, masking those which are near the edge, centre, or bright stars. The orange histogram is instead from circles placed evenly (without filtering) on the grid across the shown tangent plane. The p-values are the quantiles of the central counts (verticle line) for the blue and orange histogram cases.

In [ ]:
plot_with_hist(cat, params_gr)

In [ ]:
plot_with_hist(cat, params_ri)

In [ ]:
plot_with_hist(cat, params_ri_sgs)

In [ ]:
plot_with_hist(cat, params_gri)

In [ ]:
plot_with_hist(cat_psf, get_default_params(mode="gri", rcol="R_MAG_PSF", gcol="G_MAG_PSF", icol="I_MAG_PSF"))

In [ ]:
plot_with_hist(cat_julen, get_default_params(mode="gri",))

In [ ]:
plot_with_hist(cat_julen_stack, get_default_params(mode="gri",))

In [ ]:
plot_with_hist(cat_julen_sep, get_default_params(mode="gri", flags_max=None, flags_weight_max=None, r23_max_sigma=None))

## Parameter variations

In [ ]:
def plot_parameter_variations(params_dict):
    fig, axs = plt.subplots(len(params_dict), 1, figsize=(2, len(params_dict)*2))

    
    for i, (name, params) in enumerate(params_dict.items()):
        plt.sca(axs[i])
        cat_filt = filter_utils.filter_catalog(cat, params)
        filter_utils.plot_2d_counts(cat_filt, params.r_cen)
        plt.title(name)

In [ ]:
params_dict.keys()

In [ ]:
dm_0 = obs_props["dm"]

params_dict = {}
for delta_dm in np.arange(-1, 2, 0.5):
    dm = dm_0 + delta_dm
    params = get_default_params(dm=dm, mode="gri")
    params_dict[f"distance modulus = {dm:0.2f}"] = params


plot_parameter_variations(params_dict)

In [ ]:
ana_utils.get_extinction(0.2)

In [ ]:
A_V_0 = 0

params_dict = {}

for delta_A_V in np.arange(-0.2, 0.5, 0.1):
    A_V = A_V_0 + delta_A_V
    params = get_default_params(A_V=A_V)
    params_dict[f"A_v = {A_V}"] = params


plot_parameter_variations(params_dict)

In [ ]:


for col in mag_choices:
    params = filter_params(
        gcol = f"G_{col}",
        rcol = f"R_{col}",
        icol = f"I_{col}",
    )
    # cat_filt = filter_catalog(cat, params)
    # plot_2d_counts(cat_filt, params.r_cen, vmax=40)

    plot_with_hist(cat, params)

    plt.gcf().suptitle(f"magnitude = {col}")
    plt.show()

In [ ]:
params_dict = {}
for iso_fe_h in filter_utils.isos_fe_h:
    params = get_default_params(iso_fe_h = iso_fe_h)
    params_dict[iso_fe_h] = params


plot_parameter_variations(params_dict)

In [ ]:
params_dict = {}
for iso_age in np.arange(9.7, 10.3, 0.1):
    params = get_default_params(iso_log_age = iso_age)
    params_dict[f"iso age = {10**iso_age/1e9:0.1f} Gyr"] = params


plot_parameter_variations(params_dict)

In [ ]:
params_dict = {
    "none": get_default_params(),
    "ell": get_default_params(ellipticity_max = 0.2),
    "fwhm": get_default_params(fwhm_max = 6),
    "r_half": get_default_params(r_half_max = 6),
    "r23": get_default_params(r23_max_sigma = 3)
}
    
plot_parameter_variations(params_dict)

# Running the MCMC samples

In [ ]:
mag_choices = ["MAG_AUTO", "MAG_APER_1", "MAG_APER_2", "MAG_APER_3", "MAG_ISO", "MAG_APER_4"]
for mag in mag_choices:
    filter_utils.calibrate_mag_col(cat, mag)
    cat.rename_column`

In [ ]:
df = pd.DataFrame()

for i in range(300):

    if i % 20 == 0:
        print("step ", i)

    col = np.random.choice(mag_choices)
    params = filter_params(
        detection_sigma = np.random.choice([1.5, 2, 2.5, 3, 6]),
        r_cen = np.random.uniform(10, 80)/60,
        A_V = obs_props["A_V"] + np.random.normal(0, 0.2),
        dm = obs_props["dm"] + np.random.normal(0, 0.2),
        flags_max = np.random.choice([0, 4, 8, 1024]),
        flags_weight_max = np.random.choice([0, 2, 1024]),
        iso_fe_h = np.random.choice(filter_utils.isos_fe_h),
        iso_log_age = np.random.choice(np.arange(9.4, 10.3, 0.1)),
        iso_width = np.random.uniform(0, 0.1),
        mode = np.random.choice([ "ri",  "gr", "n"]),
        gcol = f"G_{col}",
        rcol = f"R_{col}",
        icol = f"I_{col}",
        A_i_extra = np.random.normal(0, 0.15),
    )
    d = filter_utils.catalog_stats(cat, params)

    df = pd.concat([df, d], ignore_index=True)

In [ ]:
df["significance"] = df["delta"] / df["delta_err"]
df.loc[~np.isfinite(df["significance"]), "significance"] = 0

In [ ]:
plt.scatter(df["significance"], df["sigma"] / df["sigma_err"], c=np.log10(df["Ncen"]), s=2, lw=0)
plt.axhline(2)
plt.axvline(2)
plt.xlabel("significance (bootstrap)")
plt.ylabel("significance (poisson/mean)")
plt.colorbar(label="log num members")

In [ ]:
plt.scatter(df["significance"], df["pval"], c=np.log10(df["delta"]))

plt.yscale('log')
plt.gca().invert_yaxis()

plt.ylabel("pvalue")
plt.xlabel("significance")
plt.colorbar(label="delta (overdensity)")

In [ ]:
plt.scatter(df["Ncen"], df["significance"], c=np.log10(df["pval"]))

plt.xscale('log')

plt.ylabel("significance")
plt.xlabel("number of stars in centre")
plt.colorbar(label="pvalue")

In [ ]:
for col in df.columns:
    if col not in ["sigma", "sigma_err", "Ncen", "Ntot", "Nbkg_med", "Nbkg_std", "pval", "delta", "delta_err", "significance"]:
        plt.figure(figsize=(4, 2))
        if len(df[col].unique()) == 1:
            continue
        elif len(df[col].unique()) < 20:
            groups = df.groupby(col).groups
            plt.boxplot([df.iloc[idx]["significance"] for idx in groups.values()], tick_labels=groups.keys(), )
            plt.xlabel(col)
            plt.ylabel("significance")
            plt.xticks(rotation=90)

        else:
            plt.scatter(df[col], df["significance"], c=-np.log10(df["pval"]), s=3)
            plt.xlabel(col)
            plt.ylabel("significance")
        # plt.gca().invert_yaxis()
        # plt.yscale("log")
            plt.colorbar(label="-log pvalue")
        plt.show()

In [ ]:
def retrieve_params(df_row):
    kwargs = {key: df_row[key] for key in filter_params.__dataclass_fields__.keys()}

    return filter_params(**kwargs)

In [ ]:
df.sort_values("significance")

In [ ]:
plt.hist(np.log10(df["pval"][df.pval > 0]))

In [ ]:
plt.scatter(np.sqrt(df.Ntot) * df.area_scale, df.Nbkg_std)
plt.plot([0, 20], [0, 20], color="k")

In [ ]:
plt.scatter(df.Ntot_scaled, df.Nbkg_med)
plt.plot([0, 300], [0, 300], color="k")

In [ ]:
plt.hist(df.significance, bins=np.linspace(-3, 5, 40), histtype="step", density=True, label="bootstrap")

plt.hist(df.sigma / df.sigma_err, bins=np.linspace(-3, 5, 40), histtype="step", density=True, label="average + poisson")
plt.xlabel("significance")
plt.ylabel("density")

plt.legend()

In [ ]:
params_best = retrieve_params(df.iloc[np.argmax(df.significance)])

In [ ]:
params_best

In [ ]:
catalog_stats(cat, params_best)

# Better plots

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
params_best

In [ ]:
df_sorted = df.sort_values("delta")[::-1]
df_sorted = df_sorted[np.isfinite(df.delta)]

In [ ]:
10**10.1

In [ ]:
df_sorted

The best examples

In [ ]:
for i in range(10):
    params = retrieve_params(df_sorted.iloc[i, :])
    plot_riso(cat, params, results=df_sorted.iloc[i, :])


The worst examples

In [ ]:
for i in range(10):
    d = df_sorted.iloc[-i-1, :]
    params = retrieve_params(d)
    plot_riso(cat, params, results=d)

# Sanity checks

In [ ]:
samples = [rand_overdensity(cat, 40/60, xymax=3) for i in  range(10_000)]

In [ ]:
sample_sigma = [x[0] for x in samples]

In [ ]:
sample_err = [x[1] for x in samples]

In [ ]:

plt.hist(samples)
plt.axvline(inner_overdensity(cat, 40/60)[3], color="C1")
plt.xlabel("counts")

In [ ]:
plt.hist((sample_sigma) / np.array(sample_err), density=True)
x = np.linspace(-5, 5, 1000)
y = 1/np.sqrt(2*np.pi) * np.exp(-x**2 / 2)
plt.plot(x, y)

In [ ]:
cat_test = pd.DataFrame(
    dict(
        xi = np.random.uniform(-3, 3, 2000),
        eta = np.random.uniform(-3, 3, 2000)
    )
)